# 03. 별도재무제표 수집 (감사보고서 기준)

**이 노트북이 하는 일**: **감사보고서 원문**의 재무제표 표에서 재무 값을 읽고, 사업보고서 API 값과 대조합니다. 배당에 관한 사항 표의 (별도) 항목도 모읍니다. **연결재무제표 값으로 대체하지 않습니다**.

- 준비물: OpenDART 인증키, 01 담당자에게 받은 `company_map.csv`
- 결과: 내 드라이브 `ONE_PICK_결과/`에 zip 하나(`별도_수집결과.csv`, `별도_원문확인목록.csv`, `회사별/*.csv`)
- 순서: 01 매핑표 → **03_별도재무제표** (02·03·04는 서로 독립)

**실행 방법**: 아래 **0. 설정**만 확인하고 메뉴 **런타임 → 모두 실행**. 중간에 드라이브 연결 허용 창과 키 입력 칸이 뜨면 답하면 됩니다.
KIND 목록이나 `company_map.csv`는 내 드라이브 `ONE_PICK_결과/`에 넣어 두거나 코랩 왼쪽 📁에 올리면 자동으로 찾습니다(없으면 업로드 창이 뜸).
연결·키에 문제가 있으면 **2번에서 멈추고** 고칠 방법을 알려 줍니다.

**결과 넘기기**: zip은 **내 드라이브 `ONE_PICK_결과/`**에 생깁니다(폴더가 없으면 만듦). 팀에 넘길 때는 그 zip을 공유 폴더나 메신저에 직접 올려 주세요.


| 상태 | 뜻 | 할 일 |
|---|---|---|
| 정상 | API에서 값을 찾음 | 그대로 사용 |
| 원문에서 찾음 | API에는 없고 공시 원문 표에서 찾음 | 원문 링크로 한 번 더 확인 |
| 보관본 사용 | API 대신 미리 받아 둔 사본 | 받은 날짜 확인 |
| 데이터 없음 | 원천이 '값 없음'이라고 응답(`013`, `'-'`) | 값을 채우지 말고 '확인 필요'로 둠 |
| 조회 실패 | 키·한도·점검·네트워크 문제 | 나중에 이 노트북을 다시 실행 |
| 확인 불가 | 모든 경로를 찾았지만 없거나 후보가 여러 개 | 원문 확인 목록으로 넘김 |

값이 비면 **0이나 추정값으로 채우지 않습니다.** 메모와 시도 경로를 그대로 남깁니다.

## 받는 값과 규칙 (재무회계 검토 반영)

- `당기순이익_전체`는 별도 순이익입니다. **연결 미작성 회사의 NI**로 씁니다. 연결 작성 회사의 NI는 `02_연결` 노트북의 `지배주주순이익`입니다.
- `배당금지급_지급기준`은 현금흐름표에서 그해 **실제로 나간 돈**입니다. 이익배당금액 D(결의 기준)와 다르므로 D 대신 쓰지 않습니다. 음수로 오므로 절댓값으로 저장합니다.
- `지배주주순이익`·`지배주주자본`은 별도재무제표에 없는 개념이라 받지 않습니다. 자본총계로 대체하지 않습니다.

**감사보고서 기준**: 재무 값은 감사보고서 원문 표에서 읽고(상태 '원문에서 찾음'), 같은 값을 사업보고서 API로도 받아 `대조` 칸에 적습니다.
`일치`(표 단위 안에서 같음) · `불일치(차이 n원)` · `API 값 없음` · `감사보고서 값 없음`. 불일치·값 없음·전기 값은 원문 확인 목록으로 갑니다.

## 0. 설정 (여기만 확인)

In [ ]:
# @title 0. 설정
EXPORT_FOLDER = "ONE_PICK_결과"  # @param {type:"string"}
MAP_FILE = "company_map.csv"  # @param {type:"string"}
NB_NAME = "03_별도재무제표"
YEARS = "2023,2024,2025"  # @param {type:"string"}

## 1. 코드 준비 (손댈 것 없음)

In [ ]:
# @title 1-1. 수집 코드 파일 만들기
from pathlib import Path
Path("core/sources").mkdir(parents=True, exist_ok=True)
Path("core/__init__.py").write_text("")

In [ ]:
%%writefile core/sources/__init__.py
"""외부 자료 수집과 결측 보완.

화면에는 보이는데 API가 빈 값을 주는 경우를 줄이고, 끝까지 못 찾은 값은 채우지 않고 사유와 함께 남긴다.

- base: 결측 상태, 숫자 변환, 재시도·원본 캐시, 출처 기록, 이어받기
- opendart: OpenDART (재무·배당·공시 검색·원문·주식 총수·최대주주)
- krx: KRX 수정주가·시장지수 (pykrx)
- law: 국가법령정보센터 법령 원문
- kind: KIND 고배당기업 목록 엑셀
"""

In [ ]:
%%writefile core/sources/base.py
"""수집 공통 부품: 결측 상태, 숫자 변환, 재시도·원본 캐시, 출처 기록, 이어받기.

원칙: 값을 못 찾으면 0이나 추정값으로 채우지 않는다. 왜 없는지(상태)와 어디까지 찾아봤는지(시도 경로)를 남긴다.
"""
import hashlib
import json
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable

import pandas as pd

OK = "정상"
NO_DATA = "데이터 없음"  # 원천이 '값 없음'이라고 응답함
API_ERROR = "조회 실패"  # 키·한도·점검·네트워크 문제. 데이터가 없다는 뜻이 아님
FROM_ORIGINAL = "원문에서 찾음"  # API에는 없고 공시 원문 표에서 찾음. 사람이 원문을 한 번 더 확인
STORED = "보관본 사용"  # API 대신 미리 저장해 둔 사본을 씀
UNVERIFIED = "확인 불가"  # 모든 경로를 시도했지만 값이 없거나 하나로 정해지지 않음

STATUSES = [OK, NO_DATA, API_ERROR, FROM_ORIGINAL, STORED, UNVERIFIED]

# 캐시 파일 이름·출처 기록에 남기지 않는 비밀 파라미터
SECRET_PARAMS = ("crtfc_key", "OC", "serviceKey")

DEFAULT_CACHE_DIR = Path("data/raw")  # .gitignore에 들어 있어 커밋되지 않음


@dataclass(frozen=True)
class Source:
    """값 하나의 출처. params에는 비밀값을 넣지 않는다."""

    api: str
    params: dict = field(default_factory=dict)
    rcept_no: str | None = None
    url: str | None = None
    note: str = ""


@dataclass
class Fetched:
    """수집 결과 하나. value가 None이면 status가 이유를 말한다.

    unit: value의 단위(원). 백만 원 단위 값이면 1_000_000. 원 단위로 바꾸려면 value × unit.
    attempts: 시도한 경로와 결과를 순서대로 적은 기록.
    """

    value: Any
    status: str
    source: Source | None = None
    attempts: list[str] = field(default_factory=list)
    unit: int = 1

    @property
    def found(self) -> bool:
        return self.value is not None and self.status in (OK, FROM_ORIGINAL, STORED)

    @property
    def won(self) -> int | float | None:
        """원 단위 값. 숫자가 아니면 None."""
        if isinstance(self.value, (int, float)) and not isinstance(self.value, bool):
            return self.value * self.unit
        return None


def public_params(params: dict) -> dict:
    return {k: v for k, v in params.items() if k not in SECRET_PARAMS}


# --- 비밀키 입력 ---

_SECRETS: dict[str, str] = {}  # 이번 실행 동안만 메모리에 둔다. 파일에 쓰지 않는다.


def _colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except Exception:  # noqa: BLE001 - 보안 비밀이 없거나 접근 허용 전
        return None


def secret(name: str, prompt: str | None = None, ask: Callable[[str], str] | None = None) -> str:
    """API 키를 화면에 보이지 않게 받는다.

    순서: 이번 실행에서 이미 받은 값 → 환경변수 name → Colab 보안 비밀 name → 숨김 입력(getpass).
    받은 키는 메모리에만 두며 출력·캐시 파일·출처 기록에 남기지 않는다.
    """
    import os

    if name in _SECRETS:
        return _SECRETS[name]
    value = os.environ.get(name) or _colab_secret(name)
    if not value:
        if ask is None:
            from getpass import getpass as ask
        value = ask(prompt or f"{name} 입력(화면에 보이지 않음): ")
    value = (value or "").strip()
    if not value:
        raise ValueError(f"{name}가 비어 있습니다.")
    _SECRETS[name] = value
    return value


def forget_secret(name: str) -> None:
    """잘못 입력한 키를 지워 다음 secret() 호출에서 다시 묻게 한다."""
    _SECRETS.pop(name, None)


# --- 숫자·이름 정리 ---

_EMPTY = {"", "-", "−", "–", "N/A", "NA", "nan", "None", "해당없음", "해당사항없음"}


def to_number(raw) -> int | float | None:
    """공시 숫자 문자열 → int 또는 float. 비어 있거나 '-'이면 None (0으로 바꾸지 않는다).

    '1,234' → 1234 · '△1,234'·'(1,234)'·'-1,234' → -1234 · '12.5' → 12.5 · '12.5%' → 12.5
    """
    if raw is None:
        return None
    if isinstance(raw, bool):
        return None
    if isinstance(raw, (int, float)):
        return None if raw != raw else raw  # NaN은 None
    s = re.sub(r"\s", "", str(raw)).replace(",", "").rstrip("%")
    if s in _EMPTY:
        return None
    negative = False
    if s[0] in "△▲":
        negative, s = True, s[1:]
    elif s.startswith("(") and s.endswith(")"):
        negative, s = True, s[1:-1]
    if s[:1] in ("-", "−"):
        negative, s = not negative, s[1:]
    try:
        value = float(s) if "." in s else int(s)
    except ValueError:
        return None
    return -value if negative else value


def squash(text: str) -> str:
    """비교용 이름: 공백과 가운뎃점을 모두 지운다. '현금ㆍ현물 배당결정' → '현금현물배당결정'."""
    return re.sub(r"[\sㆍ·•‧]", "", str(text or ""))


def strip_brackets(text: str) -> str:
    """비교용 이름에서 괄호 덧말과 앞 번호를 지운다. 'Ⅰ. 당기순이익(손실)' → '당기순이익'."""
    s = re.sub(r"\([^)]*\)|\[[^\]]*\]", "", squash(text))
    return re.sub(r"^([0-9]+|[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[가-하])\.", "", s)


_UNITS = {"백만원": 1_000_000, "천원": 1_000, "억원": 100_000_000, "원": 1}


def unit_from_label(label: str) -> int:
    """항목 이름 괄호 속 단위 → 원 단위 배수. '현금배당금총액(백만원)' → 1_000_000. 없으면 1."""
    for inner in re.findall(r"\(([^)]*)\)", squash(label)):
        if inner in _UNITS:
            return _UNITS[inner]
    return 1


# --- 재시도와 원본 캐시 ---

Fetch = Callable[[str, dict], bytes]


class FetchError(Exception):
    """재시도해도 응답을 받지 못함."""


def http_fetch(url: str, params: dict, timeout: float = 20) -> bytes:
    import requests

    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.content


def scrub(text: str) -> str:
    """오류 메시지 속 URL에 섞인 키를 가린다. 'crtfc_key=abc&' → 'crtfc_key=***&'."""
    return re.sub(rf"({'|'.join(SECRET_PARAMS)})=[^&\s'\"]+", r"\1=***", str(text))


def _is_client_error(error: Exception) -> bool:
    status = getattr(getattr(error, "response", None), "status_code", None)
    return status is not None and 400 <= status < 500


class Client:
    """재시도와 원본 캐시를 붙인 GET.

    - 네트워크 오류·5xx는 backoff × 2^n초 쉬고 다시 시도한다. 4xx는 다시 시도하지 않는다.
    - cache_dir가 있으면 응답 원본을 저장해 두고 다음에는 저장본을 쓴다(호출 한도 절약, 결측 원인 다시 보기).
      keep(응답)이 False면 저장하지 않는다(한도 초과·점검 응답처럼 다시 받아야 하는 경우).
    - 이번 실행 동안은 remember(응답)가 True인 응답을 메모리에 기억해 같은 요청을 다시 보내지 않는다.
      remember를 주지 않으면 keep을 따른다('데이터 없음'처럼 파일로는 남기지 않지만 같은 실행에서는 다시 물을 필요 없는 응답).
    - fetch를 바꿔 끼우면 인터넷 없이 테스트할 수 있다.
    """

    def __init__(
        self,
        fetch: Fetch | None = None,
        cache_dir: Path | str | None = None,
        retries: int = 3,
        backoff: float = 1.0,
        sleep: Callable[[float], None] = time.sleep,
    ):
        self.fetch = fetch or http_fetch
        self.cache_dir = Path(cache_dir) if cache_dir else None
        self.retries = retries
        self.backoff = backoff
        self.sleep = sleep
        self.calls = 0  # 실제로 원천에 보낸 요청 수
        self._memo: dict[str, bytes] = {}

    @staticmethod
    def _key(url: str, params: dict) -> str:
        return json.dumps([url, sorted(public_params(params).items())], ensure_ascii=False, default=str)

    def cache_path(self, url: str, params: dict) -> Path | None:
        if not self.cache_dir:
            return None
        digest = hashlib.sha256(self._key(url, params).encode()).hexdigest()[:20]
        folder = re.sub(r"[^0-9A-Za-z]+", "_", url.split("://")[-1]).strip("_")
        return self.cache_dir / folder / f"{digest}.bin"

    def get(self, url: str, params: dict, *, keep: Callable[[bytes], bool] | None = None,
            remember: Callable[[bytes], bool] | None = None) -> bytes:
        key = self._key(url, params)
        if key in self._memo:
            return self._memo[key]
        path = self.cache_path(url, params)
        if path and path.exists():
            return path.read_bytes()
        last_error: Exception | None = None
        for attempt in range(self.retries):
            try:
                self.calls += 1
                data = self.fetch(url, params)
                break
            except Exception as error:  # noqa: BLE001 - 원천마다 예외 종류가 달라 모두 받는다
                last_error = error
                if _is_client_error(error) or attempt == self.retries - 1:
                    raise FetchError(scrub(f"{url} 요청 실패({attempt + 1}번 시도): {error}")) from None
                self.sleep(self.backoff * 2**attempt)
        else:  # retries가 0인 경우
            raise FetchError(scrub(f"{url} 요청 실패: {last_error}"))
        if path and (keep is None or keep(data)):
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)
        remember = remember or keep
        if remember is None or remember(data):
            self._memo[key] = data
        return data

    def forget(self, url: str, params: dict) -> None:
        """저장본을 지워 다음에 원천에서 다시 받게 한다."""
        self._memo.pop(self._key(url, params), None)
        path = self.cache_path(url, params)
        if path and path.exists():
            path.unlink()


# --- 이어받기 (하루 호출 한도) ---


class LimitReached(Exception):
    """하루 호출 한도를 넘음. 다음 날 같은 체크포인트로 이어서 돌린다."""


def run_resumable(keys: Iterable, task: Callable[[Any], Any], checkpoint: Path | str) -> dict:
    """keys마다 task(key)를 부르고, 끝난 key를 checkpoint(JSON)에 적는다.

    LimitReached가 나면 멈추고 남은 key를 돌려준다. 다음에 같은 checkpoint로 부르면 끝난 key는 건너뛴다.
    결과 자체는 저장하지 않는다(응답 원본은 Client 캐시에 남으므로 다시 불러도 호출이 늘지 않는다).
    """
    checkpoint = Path(checkpoint)
    done: set[str] = set(json.loads(checkpoint.read_text())) if checkpoint.exists() else set()
    keys = list(keys)
    results: dict = {}
    stopped = False
    for key in keys:
        if str(key) in done:
            continue
        try:
            results[key] = task(key)
        except LimitReached:
            stopped = True
            break
        done.add(str(key))
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        checkpoint.write_text(json.dumps(sorted(done), ensure_ascii=False))
    return {
        "results": results,
        "remaining": [k for k in keys if str(k) not in done],
        "stopped_by_limit": stopped,
    }


# --- 수집 현황표 ---


def status_table(items: dict[str, Fetched]) -> pd.DataFrame:
    """항목 이름 → Fetched를 한 표로. 팀 수집 현황표·원문 확인 목록에 쓴다."""
    rows = []
    for name, f in items.items():
        rows.append({
            "항목": name,
            "값": f.value if not isinstance(f.value, pd.DataFrame) else f"표 {len(f.value)}행",
            "단위(원)": f.unit,
            "상태": f.status,
            "출처 API": f.source.api if f.source else None,
            "접수번호": f.source.rcept_no if f.source else None,
            "메모": f.source.note if f.source else "",
            "시도 경로": " → ".join(f.attempts),
        })
    # dtype=object: 정수 값이 결측 행 때문에 실수(4.8e+09)로 바뀌지 않게
    return pd.DataFrame(rows, columns=["항목", "값", "단위(원)", "상태", "출처 API", "접수번호", "메모", "시도 경로"], dtype=object)

In [ ]:
%%writefile core/sources/opendart.py
"""OpenDART 수집과 결측 보완.

화면(DART 뷰어)에는 보이는데 API가 빈 값을 주는 흔한 원인과 이 모듈의 대응:
- status 013(데이터 없음)·020(한도 초과)을 빈 표로 처리 → 상태를 나눠 기록, 020은 LimitReached로 멈춤
- 연결재무제표(CFS)가 없는 회사 → 별도(OFS)로 다시 요청하고 어느 쪽인지 남김(계정마다 허용한 쪽만)
- 계정 이름이 회사마다 다름('당기순이익(손실)', '분기순이익') → account_id 먼저, 다음에 정리한 이름으로 찾음
- 그해 보고서에 값이 없음 → 다음 해 사업보고서의 전기 값으로 찾고 '재작성됐을 수 있음'을 남김
- '-'·'△'·쉼표 → 숫자 변환에서 결측과 음수를 구분
- 정정공시 → 공시 검색은 최종본만(last_reprt_at=Y)
- API 범위 밖(배당 결정 공시 숫자 등) → 공시 원문(document.xml) 표에서 찾고 '원문에서 찾음'으로 표시
- 감사보고서 기준 재무 값 → 감사보고서·연결감사보고서 원문의 재무제표 표를 읽음(audit_account)

끝까지 못 찾으면 0으로 채우지 않고 '확인 불가'와 DART 뷰어 링크를 남긴다.
"""
import html
import io
import json
import re
import xml.etree.ElementTree as ET
import zipfile
from dataclasses import dataclass
from datetime import date, datetime

import pandas as pd

from core.sources.base import (
    API_ERROR,
    FROM_ORIGINAL,
    NO_DATA,
    OK,
    UNVERIFIED,
    Client,
    Fetched,
    LimitReached,
    Source,
    public_params,
    secret,
    squash,
    strip_brackets,
    to_number,
    unit_from_label,
)

BASE = "https://opendart.fss.or.kr/api"
KEY_NAME = "DART_API_KEY"

REPORT_CODES = {"11011": "사업보고서", "11012": "반기보고서", "11013": "1분기보고서", "11014": "3분기보고서"}
FS_NAMES = {"CFS": "연결", "OFS": "별도"}

STATUS_MESSAGES = {
    "000": "정상",
    "010": "등록되지 않은 키",
    "011": "사용할 수 없는 키",
    "012": "접근할 수 없는 IP",
    "013": "조회된 데이터 없음",
    "014": "파일이 존재하지 않음",
    "020": "요청 제한 초과(하루 20,000건)",
    "021": "조회 가능한 회사 개수 초과(최대 100건)",
    "100": "필드의 부적절한 값",
    "101": "부적절한 접근",
    "800": "시스템 점검 중",
    "900": "정의되지 않은 오류",
    "901": "사용자 계정의 개인정보 보유기간 만료",
}
KEY_ERRORS = {"010", "011", "012", "901"}  # 키를 고치기 전에는 어떤 요청도 성공하지 않음


class DartKeyError(Exception):
    """키·IP 문제. 비밀키를 다시 입력해야 한다(base.forget_secret 후 다시 실행)."""


def viewer_url(rcept_no: str) -> str:
    """사람이 원문을 확인할 DART 뷰어 주소."""
    return f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}"


@dataclass(frozen=True)
class Account:
    """찾을 계정.

    ids: 표준 계정코드(account_id). 감사보고서 원문에는 없어 이름으로만 찾는다.
    names: 계정 이름 후보. 비교할 때 양쪽 모두 strip_brackets로 공백·괄호 덧말을 지운다.
    statements: sj_div (BS 재무상태표, IS 손익계산서, CIS 포괄손익계산서, CF 현금흐름표)
    fs_only: 이 재무제표에서만 받는다. 다른 쪽을 요청하면 대체하지 않고 '확인 불가'
    absolute: 음수로 오는 값을 절댓값으로 저장한다
    """

    ids: tuple[str, ...]
    names: tuple[str, ...]
    statements: tuple[str, ...]
    fs_only: tuple[str, ...] = ("CFS", "OFS")
    absolute: bool = False


# 재무회계 팀원 검토(2026-10-22) 반영. 자동 대체는 하지 않는다.
ACCOUNTS = {
    # 배당성향 분모(NI). 연결 작성 회사는 CFS에서만 가져옴
    # "지배기업소유주지분" 단독 이름은 CIS 총포괄이익 귀속 행과 겹쳐서 제외
    "지배주주순이익": Account(
        ids=("ifrs-full_ProfitLossAttributableToOwnersOfParent",),
        names=("지배기업소유주지분순이익", "지배기업소유주지분당기순이익",
               "지배기업의소유주에게귀속되는당기순이익", "지배기업소유주에게귀속되는당기순이익",
               "지배기업의소유주에게귀속되는당기순이익(손실)", "지배기업소유주에게귀속되는당기순이익(손실)",
               "지배주주지분순이익", "지배주주순이익"),
        statements=("IS", "CIS"),
        fs_only=("CFS",),
    ),
    # CFS: 비지배지분 포함 연결 전체 순이익 / OFS: 별도 순이익(연결 미작성 회사의 NI)
    # 배당 표(DIVIDEND_ITEMS)의 "당기순이익"과는 다른 값
    "당기순이익_전체": Account(
        ids=("ifrs-full_ProfitLoss",),
        names=("당기순이익", "당기순이익(손실)", "당기순손익", "당기순손실", "연결당기순이익",
               "분기순이익", "반기순이익"),
        statements=("IS", "CIS"),
    ),
    "부채총계": Account(
        ids=("ifrs-full_Liabilities",),
        names=("부채총계",),
        statements=("BS",),
    ),
    # 비지배지분 포함. 2배 기준의 '자본총액'이 이 값인지 지배주주자본인지는 시행규칙 원문 확인 필요
    "자본총계": Account(
        ids=("ifrs-full_Equity",),
        names=("자본총계",),
        statements=("BS",),
    ),
    # CFS에만 있음. OFS에서 없을 때 자본총계로 대체하지 않음
    "지배주주자본": Account(
        ids=("ifrs-full_EquityAttributableToOwnersOfParent",),
        names=("지배기업소유주지분", "지배기업의소유주지분", "지배기업의소유주에게귀속되는자본"),
        statements=("BS",),
        fs_only=("CFS",),
    ),
    # 지급 기준(그해 실제 나간 돈). D(이익배당금액, 결의 기준)와 다름. 자동 대체 금지
    # OFS 현금흐름표에서만 가져옴. 음수로 오므로 저장할 때 절댓값으로 맞춤
    "배당금지급_지급기준": Account(
        ids=("ifrs-full_DividendsPaidClassifiedAsFinancingActivities",
             "ifrs-full_DividendsPaidClassifiedAsOperatingActivities",
             "ifrs-full_DividendsPaidToEquityHoldersOfParentClassifiedAsFinancingActivities",
             "ifrs-full_DividendsPaid"),
        names=("배당금의지급", "배당금지급", "현금배당금의지급", "배당금지급액"),
        statements=("CF",),
        fs_only=("OFS",),
        absolute=True,
    ),
}

# 배당에 관한 사항(alotMatter)의 se 항목. strip_brackets로 정리한 이름과 같은지 본다.
DIVIDEND_ITEMS = ("현금배당금총액", "주당현금배당금", "현금배당성향", "현금배당수익률", "당기순이익",
                  "주당순이익", "주식배당금총액", "주당주식배당", "주당액면가액")

PERIODS = {"thstrm": "당기", "frmtrm": "전기", "lwfr": "전전기"}
CORP_CLS = {"Y": "유가", "K": "코스닥", "N": "코넥스", "E": "기타"}  # 기업개황 법인구분
AUDIT_REPORTS = {"CFS": ("F002", "연결감사보고서"), "OFS": ("F001", "감사보고서")}  # (공시 상세유형, 보고서 이름)
NO_ROW = "행 없음"  # pick_dividend 안에서만 쓰는 상태: 다음 보고서로 넘어가 찾는다


def _fiscal_year(value) -> str:
    return str(int(value))


class Dart:
    """OpenDART 요청 묶음. key를 주지 않으면 비밀 입력(base.secret)으로 받는다."""

    def __init__(self, key: str | None = None, client: Client | None = None):
        self._key = key
        self.client = client or Client()
        self._audit_cache: dict = {}  # (corp_code, 사업연도, fs) → 읽은 감사보고서 재무제표

    @property
    def key(self) -> str:
        if not self._key:
            self._key = secret(KEY_NAME, "OpenDART 인증키 입력(화면에 보이지 않음): ")
        return self._key

    # --- 요청 공통 ---

    def call(self, endpoint: str, params: dict) -> tuple[str, str, dict]:
        """JSON API 한 번. (status, message, 본문)을 돌려준다. 키 문제·한도 초과는 예외로 멈춘다."""
        url = f"{BASE}/{endpoint}"
        full = {"crtfc_key": self.key, **params}
        raw = self.client.get(url, full, keep=lambda b: _json_status(b) == "000",
                              remember=lambda b: _json_status(b) in ("000", "013"))
        body = json.loads(raw)
        status = str(body.get("status", ""))
        message = body.get("message") or STATUS_MESSAGES.get(status, "")
        if status in KEY_ERRORS:
            raise DartKeyError(f"{status} {STATUS_MESSAGES.get(status, message)}")
        if status == "020":
            raise LimitReached(f"OpenDART {STATUS_MESSAGES['020']}")
        return status, message, body

    def _rows(self, endpoint: str, params: dict) -> tuple[pd.DataFrame | None, str, str]:
        """list 응답을 표로. 실패하면 (None, 상태, 사유)."""
        status, message, body = self.call(endpoint, params)
        label = f"{endpoint} {' '.join(str(v) for v in params.values())}"
        if status == "000":
            return pd.DataFrame(body.get("list", [])), OK, f"{label}: 정상"
        if status == "013":
            return None, NO_DATA, f"{label}: 013 데이터 없음"
        return None, API_ERROR, f"{label}: {status} {message}"

    # --- 고유번호 ---

    def corp_codes(self) -> pd.DataFrame:
        """전체 회사 고유번호표(corp_code, corp_name, stock_code, modify_date). 상장사만 stock_code가 있다."""
        url = f"{BASE}/corpCode.xml"
        raw = self.client.get(url, {"crtfc_key": self.key}, keep=lambda b: b[:2] == b"PK")
        if raw[:2] != b"PK":
            status = _xml_status(raw)
            if status in KEY_ERRORS:
                raise DartKeyError(f"{status} {STATUS_MESSAGES.get(status, '')}")
            if status == "020":
                raise LimitReached(f"OpenDART {STATUS_MESSAGES['020']}")
            raise ValueError(f"고유번호 파일을 받지 못함: {status} {STATUS_MESSAGES.get(status, '')}")
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            root = ET.fromstring(z.read(z.namelist()[0]))
        rows = [{c.tag: (c.text or "").strip() for c in item} for item in root.iter("list")]
        return pd.DataFrame(rows, columns=["corp_code", "corp_name", "stock_code", "modify_date"])

    # --- 재무제표 ---

    def statement(self, corp_code: str, year, reprt_code: str = "11011",
                  fs_divs: tuple[str, ...] = ("CFS", "OFS")) -> Fetched:
        """전체 재무제표(fnlttSinglAcntAll). 연결이 없으면 별도로 다시 요청한다. value는 표, fs_div 열 포함."""
        attempts: list[str] = []
        worst = NO_DATA
        for fs in fs_divs:
            params = {"corp_code": corp_code, "bsns_year": _fiscal_year(year), "reprt_code": reprt_code, "fs_div": fs}
            df, status, note = self._rows("fnlttSinglAcntAll.json", params)
            attempts.append(f"{FS_NAMES[fs]} {note.split(': ', 1)[1]}")
            if df is not None and not df.empty:
                df["fs_div"] = fs
                rcept = str(df["rcept_no"].iloc[0]) if "rcept_no" in df else None
                return Fetched(df, OK, Source("fnlttSinglAcntAll", public_params(params), rcept), attempts)
            if status == API_ERROR:
                worst = API_ERROR
        return Fetched(None, worst, None, attempts)

    def account(self, corp_code: str, year, item: str, reprt_code: str = "11011",
                field: str = "thstrm_amount", fs_divs: tuple[str, ...] = ("CFS", "OFS")) -> Fetched:
        """계정 값 하나(원 단위). 찾는 순서:

        1) 그해 전체 재무제표(fs_divs 순서, 기본 연결→별도)에서 account_id, 다음에 이름으로
        2) 그해 주요계정(fnlttSinglAcnt)에서 이름으로
        3) 다음 해 사업보고서의 전기 값(frmtrm_amount). 재작성됐을 수 있어 메모를 남긴다
        계정마다 허용한 재무제표(Account.fs_only)만 본다. 다른 계정 값으로 대체하지 않는다.
        분기 손익의 누적 값이 필요하면 field='thstrm_add_amount'.
        fs_divs=("CFS",)나 ("OFS",)로 한쪽만 주면 다른 쪽으로 넘어가지 않는다(2~4단계도 같은 쪽만 본다).
        """
        spec = ACCOUNTS[item]
        allowed = tuple(fs for fs in fs_divs if fs in spec.fs_only)
        if not allowed:
            wanted = "·".join(FS_NAMES[f] for f in fs_divs)
            only = "·".join(FS_NAMES[f] for f in spec.fs_only)
            return Fetched(None, UNVERIFIED, Source("fnlttSinglAcntAll", note=f"{item}은 {only}재무제표에서만 받음({wanted} 요청)"),
                           [f"{item}: {wanted}재무제표 대상 아님"])
        fs_divs = allowed
        attempts: list[str] = []

        st = self.statement(corp_code, year, reprt_code, fs_divs)
        attempts += st.attempts
        if st.found:
            fs = st.value["fs_div"].iloc[0]
            hit = find_account(st.value, spec, field)
            if hit:
                value, how = hit
                attempts.append(f"{item}: {how}")
                src = Source("fnlttSinglAcntAll", st.source.params, st.source.rcept_no, note=f"{FS_NAMES[fs]} · {how}")
                return _number_result(value, src, attempts, spec.absolute)
            attempts.append(f"{item}: 전체 재무제표에 계정 없음")

        main = self._main_accounts(corp_code, year, reprt_code, spec, field, attempts, fs_divs)
        if main:
            return main

        if reprt_code == "11011" and field == "thstrm_amount":
            nxt = self.statement(corp_code, int(year) + 1, "11011", fs_divs)
            attempts += [f"다음 해 {a}" for a in nxt.attempts]
            if nxt.found:
                hit = find_account(nxt.value, spec, "frmtrm_amount")
                if hit:
                    value, how = hit
                    attempts.append(f"{item}: 다음 해 전기 값 {how}")
                    src = Source("fnlttSinglAcntAll", nxt.source.params, nxt.source.rcept_no,
                                 note=f"{int(year) + 1} 사업보고서의 전기 값(재작성됐을 수 있음)")
                    return _number_result(value, src, attempts, spec.absolute)

        rcept = st.source.rcept_no if st.source else None
        status = API_ERROR if st.status == API_ERROR else UNVERIFIED
        note = f"원문 확인: {viewer_url(rcept)}" if rcept else "사업보고서 원문 확인"
        return Fetched(None, status, Source("fnlttSinglAcntAll", {"corp_code": corp_code, "bsns_year": str(year)}, rcept, note=note), attempts)

    def _main_accounts(self, corp_code, year, reprt_code, spec: Account, field, attempts,
                       fs_divs: tuple[str, ...] = ("CFS", "OFS")) -> Fetched | None:
        params = {"corp_code": corp_code, "bsns_year": _fiscal_year(year), "reprt_code": reprt_code}
        df, _, note = self._rows("fnlttSinglAcnt.json", params)
        attempts.append(f"주요계정 {note.split(': ', 1)[1]}")
        if df is None or df.empty:
            return None
        if "fs_div" not in df:
            attempts.append("주요계정: 연결·별도 구분 열 없음")
            return None
        parts = [(FS_NAMES[fs], df[df["fs_div"] == fs]) for fs in fs_divs]
        for fs_name, part in parts:
            hit = find_account(part, spec, field, use_ids=False)
            if hit:
                value, how = hit
                attempts.append(f"주요계정 {fs_name} {how}")
                rcept = str(part["rcept_no"].iloc[0]) if "rcept_no" in part else None
                return _number_result(value, Source("fnlttSinglAcnt", params, rcept, note=f"{fs_name} · {how}"), attempts,
                                      spec.absolute)
        return None

    # --- 배당에 관한 사항 ---

    def dividends(self, corp_code: str, year, reprt_code: str = "11011") -> Fetched:
        """배당에 관한 사항(alotMatter) 표 그대로."""
        params = {"corp_code": corp_code, "bsns_year": _fiscal_year(year), "reprt_code": reprt_code}
        df, status, note = self._rows("alotMatter.json", params)
        rcept = str(df["rcept_no"].iloc[0]) if df is not None and "rcept_no" in df and not df.empty else None
        return Fetched(df, status, Source("alotMatter", params, rcept), [note])

    def dividend_item(self, corp_code: str, year, item: str, stock_knd: str = "보통주",
                      basis: str = "연결", strict_basis: bool = True) -> Fetched:
        """배당 항목 값 하나. 단위는 항목 이름에서 읽는다(현금배당금총액은 백만 원).

        찾는 순서: 그해 사업보고서 당기 → 다음 해 전기 → 다다음 해 전전기.
        행은 있는데 값이 '-'이면 '데이터 없음'(무배당일 수 있어 배당 결정 공시로 확인), 행이 없으면 '확인 불가'.
        """
        if item not in DIVIDEND_ITEMS:
            raise ValueError(f"알 수 없는 배당 항목: {item}")
        attempts: list[str] = []
        saw_dash = False
        last_source = None
        for offset, period in ((0, "thstrm"), (1, "frmtrm"), (2, "lwfr")):
            got = self.dividends(corp_code, int(year) + offset)
            attempts += got.attempts
            if got.status == API_ERROR:
                return Fetched(None, API_ERROR, got.source, attempts)
            if not got.found or got.value.empty:
                continue
            last_source = got.source
            pick = pick_dividend(got.value, item, period, stock_knd, basis, strict_basis)
            attempts.append(f"{int(year) + offset} 보고서 {PERIODS[period]}: {pick['how']}")
            if pick["status"] == OK:
                note = pick["note"]
                if offset:
                    note = f"{int(year) + offset} 사업보고서의 {PERIODS[period]} 값" + (f" · {note}" if note else "")
                src = Source("alotMatter", got.source.params, got.source.rcept_no, note=note)
                return Fetched(pick["value"], OK, src, attempts, unit=pick["unit"])
            if pick["status"] == UNVERIFIED:
                return Fetched(None, UNVERIFIED, Source("alotMatter", got.source.params, got.source.rcept_no,
                                                        note=pick["note"]), attempts)
            saw_dash = saw_dash or pick["status"] == NO_DATA
        if saw_dash:
            note = "보고서에 '-'로 표시: 무배당일 수 있음. 배당 결정 공시로 확인"
            return Fetched(None, NO_DATA, _with_note(last_source, note), attempts)
        return Fetched(None, UNVERIFIED, _with_note(last_source, "배당 항목 행을 찾지 못함: 사업보고서 원문 확인"), attempts)

    # --- 공시 검색·원문 ---

    def disclosures(self, bgn_de, end_de, corp_code: str | None = None, keywords: tuple[str, ...] = (),
                    pblntf_ty: str | None = None, final_only: bool = True,
                    pblntf_detail_ty: str | None = None) -> Fetched:
        """공시 목록(list.json). 모든 쪽을 넘겨 받고 report_nm에 keywords가 하나라도 들어간 것만 남긴다.

        final_only=True면 정정 전 공시를 빼고 최종본만 받는다(last_reprt_at=Y).
        회사를 정하지 않으면 OpenDART가 3개월까지만 받아 주므로 기간을 3개월씩 나눠 요청한다.
        pblntf_ty: A 정기공시, B 주요사항보고, F 외부감사관련, I 거래소공시(배당 결정·기업가치 제고 계획) 등.
        pblntf_detail_ty: F001 감사보고서, F002 연결감사보고서 등.
        """
        frames, attempts = [], []
        chunks = [(_ymd(bgn_de), _ymd(end_de))] if corp_code else _quarters(_ymd(bgn_de), _ymd(end_de))
        for start, end in chunks:
            page = 1
            while True:
                params = {"bgn_de": start, "end_de": end, "page_no": page, "page_count": 100,
                          "last_reprt_at": "Y" if final_only else "N"}
                if corp_code:
                    params["corp_code"] = corp_code
                if pblntf_ty:
                    params["pblntf_ty"] = pblntf_ty
                if pblntf_detail_ty:
                    params["pblntf_detail_ty"] = pblntf_detail_ty
                status, message, body = self.call("list.json", params)
                if status == "013":
                    attempts.append(f"{start}~{end}: 013 데이터 없음")
                    break
                if status != "000":
                    attempts.append(f"{start}~{end} {page}쪽: {status} {message}")
                    return Fetched(None, API_ERROR, Source("list", public_params(params)), attempts)
                frames.append(pd.DataFrame(body.get("list", [])))
                total = int(body.get("total_page", 1) or 1)
                if page >= total:
                    attempts.append(f"{start}~{end}: {body.get('total_count', '')}건")
                    break
                page += 1
        df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["rcept_no", "report_nm"])
        if keywords and not df.empty:
            keys = [squash(k) for k in keywords]
            df = df[df["report_nm"].map(lambda n: any(k in squash(n) for k in keys))]
        df = df.reset_index(drop=True)
        if not df.empty:
            df["정정"] = df["report_nm"].map(lambda n: "정정" in str(n))
            df["원문"] = df["rcept_no"].map(viewer_url)
        status = OK if not df.empty else NO_DATA
        return Fetched(df, status, Source("list", {"bgn_de": bgn_de, "end_de": end_de, "corp_code": corp_code}), attempts)

    def document(self, rcept_no: str) -> Fetched:
        """공시 원문(document.xml). value는 {파일 이름: 본문}."""
        params = {"rcept_no": rcept_no}
        raw = self.client.get(f"{BASE}/document.xml", {"crtfc_key": self.key, **params},
                              keep=lambda b: b[:2] == b"PK")
        src = Source("document", params, rcept_no, viewer_url(rcept_no))
        if raw[:2] != b"PK":
            status = _xml_status(raw) or _json_status(raw)
            if status in KEY_ERRORS:
                raise DartKeyError(f"{status} {STATUS_MESSAGES.get(status, '')}")
            if status == "020":
                raise LimitReached(f"OpenDART {STATUS_MESSAGES['020']}")
            kind = NO_DATA if status in ("013", "014") else API_ERROR
            note = f"{status} {STATUS_MESSAGES.get(status, '')}: 뷰어에서 직접 확인"
            return Fetched(None, kind, _with_note(src, note), [f"document {rcept_no}: {note}"])
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            files = {name: _decode(z.read(name)) for name in z.namelist()}
        return Fetched(files, OK, src, [f"document {rcept_no}: 파일 {len(files)}개"])

    def from_original(self, rcept_no: str, label: str, as_text: bool = False) -> Fetched:
        """공시 원문 표에서 label 행의 값을 찾는다. 찾으면 '원문에서 찾음'(사람이 한 번 더 확인)."""
        doc = self.document(rcept_no)
        if not doc.found:
            return doc
        found = find_in_tables(doc.value, label, as_text=as_text)
        found.source = Source("document", {"rcept_no": rcept_no}, rcept_no, viewer_url(rcept_no), found.source.note)
        found.attempts = doc.attempts + found.attempts
        return found

    def dividend_decision(self, rcept_no: str) -> dict[str, Fetched]:
        """배당 결정 공시(현금ㆍ현물배당결정) 원문의 주요 숫자. 금액은 원 단위."""
        doc = self.document(rcept_no)
        labels = {"1주당배당금": ("1주당배당금", False), "배당금총액": ("배당금총액", False),
                  "시가배당율": ("시가배당율", False), "배당기준일": ("배당기준일", True)}
        out = {}
        for name, (label, as_text) in labels.items():
            if not doc.found:
                out[name] = doc
                continue
            f = find_in_tables(doc.value, label, as_text=as_text)
            f.source = Source("document", {"rcept_no": rcept_no}, rcept_no, viewer_url(rcept_no), f.source.note)
            out[name] = f
        return out

    # --- 감사보고서 원문 재무제표 ---

    def audit_report(self, corp_code: str, year, fs: str = "CFS") -> Fetched:
        """그 사업연도의 감사보고서(OFS) 또는 연결감사보고서(CFS) 공시 한 건. value는 list.json 행."""
        detail, name = AUDIT_REPORTS[fs]
        y = int(year)
        found = self.disclosures(f"{y}0101", f"{y + 1}1231", corp_code=corp_code,
                                 pblntf_ty="F", pblntf_detail_ty=detail)
        attempts = [f"{name} 검색 {a}" for a in found.attempts]
        if found.status == API_ERROR:
            return Fetched(None, API_ERROR, found.source, attempts)
        df = found.value if found.found else pd.DataFrame(columns=["report_nm", "rcept_no"])
        if not df.empty:
            plain = df["report_nm"].map(strip_brackets)
            same_kind = plain.str.startswith(name) if fs == "CFS" else plain.str.startswith(name) & ~plain.str.startswith("연결")
            df = df[same_kind & df["report_nm"].map(lambda n: f"({y}." in squash(n))]
        if df.empty:
            note = f"{y} 사업연도 {name} 없음" + (" (연결 미작성 회사일 수 있음)" if fs == "CFS" else "")
            return Fetched(None, NO_DATA, Source("list", {"corp_code": corp_code, "year": y}, note=note), attempts + [note])
        row = df.sort_values("rcept_no").iloc[-1]
        attempts.append(f"{name} {row['rcept_no']} ({row['report_nm']})")
        return Fetched(row, OK, Source("list", {"corp_code": corp_code, "year": y}, row["rcept_no"], viewer_url(row["rcept_no"])),
                       attempts)

    def audit_statement(self, corp_code: str, year, fs: str = "CFS") -> Fetched:
        """감사보고서 원문의 재무제표 표를 읽어 sj_div·account_nm·thstrm_amount·frmtrm_amount(원 단위) 표로."""
        key = (corp_code, int(year), fs)
        if key in self._audit_cache:
            return self._audit_cache[key]
        rep = self.audit_report(corp_code, year, fs)
        if not rep.found:
            result = rep
        else:
            rcept = rep.value["rcept_no"]
            doc = self.document(rcept)
            attempts = rep.attempts + doc.attempts
            src = Source("감사보고서", {"corp_code": corp_code, "year": int(year), "fs": fs}, rcept, viewer_url(rcept))
            if not doc.found:
                result = Fetched(None, doc.status, _with_note(src, doc.source.note if doc.source else ""), attempts)
            else:
                df = parse_financial_statements(doc.value)
                if df.empty:
                    result = Fetched(None, UNVERIFIED, _with_note(src, "원문에서 재무제표 표를 읽지 못함"), attempts)
                else:
                    kinds = ", ".join(sorted(df["sj_div"].unique()))
                    result = Fetched(df, FROM_ORIGINAL, src, attempts + [f"원문 재무제표 {kinds} {len(df)}행"])
        self._audit_cache[key] = result
        return result

    def audit_account(self, corp_code: str, year, item: str, fs: str = "CFS") -> Fetched:
        """감사보고서 원문 기준 계정 값(원 단위). 상태는 '원문에서 찾음'.

        찾는 순서: 그해 감사보고서 당기 칸 → 다음 해 감사보고서 전기 칸(재작성됐을 수 있음).
        감사보고서에는 account_id가 없어 이름으로만 찾는다. 계정마다 허용한 재무제표(fs_only)만 본다.
        """
        spec = ACCOUNTS[item]
        if fs not in spec.fs_only:
            only = "·".join(FS_NAMES[f] for f in spec.fs_only)
            return Fetched(None, UNVERIFIED, Source("감사보고서", note=f"{item}은 {only}재무제표에서만 받음"),
                           [f"{item}: {FS_NAMES[fs]} 대상 아님"])
        attempts: list[str] = []
        last_src = None
        for offset, column, label in ((0, "thstrm_amount", "당기"), (1, "frmtrm_amount", "전기")):
            st = self.audit_statement(corp_code, int(year) + offset, fs)
            attempts += [f"{int(year) + offset} {a}" for a in st.attempts]
            if st.status == API_ERROR:
                return Fetched(None, API_ERROR, st.source, attempts)
            if not st.found:
                last_src = last_src or st.source
                continue
            last_src = st.source
            df = st.value[st.value[column].notna()]
            unknown_unit = df[df["unit"].isna()]
            hit = find_account(df[df["unit"].notna()], spec, column, use_ids=False)
            if hit:
                value, how = hit
                row, table_unit = _source_row(df, spec, column, value)
                note = f"{FS_NAMES[fs]} {AUDIT_REPORTS[fs][1]} {label} 칸 · {how}" + (f" · {row}" if row else "")
                if offset:
                    note += f" · {int(year) + 1} 감사보고서의 전기 값(재작성됐을 수 있음)"
                params = {**st.source.params, "표 단위": table_unit}
                got = _number_result(value, Source("감사보고서", params, st.source.rcept_no, st.source.url, note),
                                     attempts + [f"{item}: {label} {how}"], spec.absolute)
                if got.status == OK:
                    got.status = FROM_ORIGINAL
                return got
            if find_account(unknown_unit, spec, column, use_ids=False):
                return Fetched(None, UNVERIFIED, _with_note(st.source, "표 단위(원·천원·백만원)를 찾지 못함: 원문 확인"),
                               attempts)
            attempts.append(f"{item}: {int(year) + offset} 감사보고서 {label} 칸에 계정 없음")
        return Fetched(None, UNVERIFIED, _with_note(last_src, f"감사보고서에서 '{item}' 값을 찾지 못함: 원문 확인"), attempts)

    # --- 기업개황 ---

    def company(self, corp_code: str) -> Fetched:
        """기업개황(company.json): 정식 회사명·종목코드·법인구분(시장)·업종코드·결산월. value는 dict."""
        params = {"corp_code": corp_code}
        status, message, body = self.call("company.json", params)
        if status != "000":
            kind = NO_DATA if status == "013" else API_ERROR
            return Fetched(None, kind, Source("company", params), [f"company {corp_code}: {status} {message}"])
        keys = ("corp_name", "stock_name", "stock_code", "corp_cls", "induty_code", "acc_mt")
        value = {k: str(body.get(k, "") or "").strip() for k in keys}
        value["시장"] = CORP_CLS.get(value["corp_cls"], value["corp_cls"])
        return Fetched(value, OK, Source("company", params), [f"company {corp_code}: 정상"])

    # --- 주식 총수·최대주주 ---

    def share_counts(self, corp_code: str, year, reprt_code: str = "11011", kind: str = "보통주") -> dict[str, Fetched]:
        """주식의 총수 현황(stockTotqySttus): 발행주식총수·자기주식수·유통주식수.

        유통주식수가 비어 있고 발행·자기주식이 있으면 발행 − 자기로 계산하고 메모를 남긴다.
        """
        params = {"corp_code": corp_code, "bsns_year": _fiscal_year(year), "reprt_code": reprt_code}
        df, status, note = self._rows("stockTotqySttus.json", params)
        fields = {"발행주식총수": "istc_totqy", "자기주식수": "tesstk_co", "유통주식수": "distb_stock_co"}
        if df is None or df.empty:
            return {k: Fetched(None, status, Source("stockTotqySttus", params), [note]) for k in fields}
        row = df[df["se"].map(lambda s: squash(s).startswith(squash(kind)[:2]))]
        rcept = str(df["rcept_no"].iloc[0]) if "rcept_no" in df else None
        src = Source("stockTotqySttus", params, rcept)
        if row.empty:
            return {k: Fetched(None, UNVERIFIED, _with_note(src, f"'{kind}' 행 없음"), [note]) for k in fields}
        r = row.iloc[0]
        out = {k: _number_result(r.get(col), src, [note]) for k, col in fields.items()}
        issued, treasury = out["발행주식총수"].value, out["자기주식수"].value
        if out["유통주식수"].value is None and issued is not None and treasury is not None:
            out["유통주식수"] = Fetched(issued - treasury, OK, _with_note(src, "발행주식총수 − 자기주식수로 계산"), [note])
        return out

    def largest_holder_stake(self, corp_code: str, year, reprt_code: str = "11011", stock_knd: str = "보통주") -> Fetched:
        """최대주주 및 특수관계인 기말 지분율 합계(%). '계' 행이 있으면 그 값, 없으면 개별 행을 더한다."""
        params = {"corp_code": corp_code, "bsns_year": _fiscal_year(year), "reprt_code": reprt_code}
        df, status, note = self._rows("hyslrSttus.json", params)
        if df is None or df.empty:
            return Fetched(None, status, Source("hyslrSttus", params), [note])
        rcept = str(df["rcept_no"].iloc[0]) if "rcept_no" in df else None
        src = Source("hyslrSttus", params, rcept)
        knd = df["stock_knd"].map(squash) if "stock_knd" in df else pd.Series("", index=df.index)
        part = df[knd.str.startswith(squash(stock_knd)[:2]) | knd.isin(["", "-"])]
        total = part[part["nm"].map(lambda n: squash(n) in ("계", "합계"))]
        if not total.empty:
            return _number_result(total["trmend_posesn_stock_qota_rt"].iloc[0], _with_note(src, "'계' 행"), [note])
        values = [to_number(v) for v in part["trmend_posesn_stock_qota_rt"]]
        values = [v for v in values if v is not None]
        if not values:
            return Fetched(None, UNVERIFIED, _with_note(src, "지분율 값 없음"), [note])
        return Fetched(round(sum(values), 2), OK, _with_note(src, f"개별 {len(values)}행 합계"), [note])


# --- 표에서 값 고르기 (요청 없이 쓰는 순수 함수) ---


def find_account(df: pd.DataFrame, spec: Account, field: str = "thstrm_amount",
                 use_ids: bool = True) -> tuple[int | float | None, str] | None:
    """재무제표 표에서 계정 값을 찾는다. (값, 찾은 방법) 또는 None.

    값이 '-'인 행은 찾은 것으로 보지 않는다. 후보 값이 서로 다르면 None을 돌려 사람이 보게 한다.
    """
    if df is None or df.empty or field not in df:
        return None
    rows = df[df["sj_div"].isin(spec.statements)] if "sj_div" in df else df
    tries = []
    if use_ids and "account_id" in rows:
        tries.append(("account_id 일치", rows[rows["account_id"].isin(spec.ids)]))
    names = rows["account_nm"].map(strip_brackets) if "account_nm" in rows else pd.Series(dtype=str)
    tries.append(("계정 이름 일치", rows[names.isin({strip_brackets(n) for n in spec.names})]))
    for how, hit in tries:
        values = [to_number(v) for v in hit[field]]
        values = sorted({v for v in values if v is not None})
        if len(values) == 1:
            return values[0], how
        if len(values) > 1:
            return None
    return None


def pick_dividend(df: pd.DataFrame, item: str, period: str = "thstrm", stock_knd: str = "보통주",
                  basis: str = "연결", strict_basis: bool = True) -> dict:
    """alotMatter 표에서 항목 하나. {'status', 'value', 'unit', 'how', 'note'}.

    status: OK · NO_DATA(행은 있으나 값이 '-') · UNVERIFIED(값이 여러 개) · NO_ROW(행 자체가 없음)

    - se 이름은 '(연결)당기순이익(백만원)'처럼 기준·단위가 붙어 있어 괄호를 떼고 비교한다.
    - 연결/별도 표시가 있는 항목은 basis 쪽만 본다. 없으면 '행 없음'(다른 쪽으로 대체하지 않음).
      strict_basis=False면 다른 쪽 값을 쓰고 메모한다(비교용으로만).
    - 주식 종류가 적힌 행은 stock_knd(보통주)만 본다.
    """
    se = df["se"].astype(str)
    rows = df[se.map(strip_brackets) == squash(item)]
    if rows.empty:
        return {"status": NO_ROW, "value": None, "unit": 1, "how": "행 없음", "note": ""}
    if "stock_knd" in rows:
        knd = rows["stock_knd"].map(squash)
        typed = knd.str.startswith(squash(stock_knd)[:2])
        if typed.any():
            rows = rows[typed]
        elif not knd.isin(["", "-", "nan", "None"]).all():
            return {"status": NO_ROW, "value": None, "unit": 1, "how": f"{stock_knd} 행 없음", "note": ""}
    note = ""
    marks = rows["se"].map(lambda s: "연결" if "(연결)" in squash(s) else "별도" if "(별도)" in squash(s) else "")
    if marks.isin(["연결", "별도"]).any():
        if (marks == basis).any():
            rows = rows[marks == basis]
        elif strict_basis:
            return {"status": NO_ROW, "value": None, "unit": 1, "how": f"({basis}) 행 없음", "note": ""}
        else:
            other = "별도" if basis == "연결" else "연결"
            rows = rows[marks == other]
            note = f"{basis} 값이 없어 {other} 값을 씀"
    raw_values = list(rows[period]) if period in rows else []
    values = sorted({v for v in (to_number(x) for x in raw_values) if v is not None})
    unit = unit_from_label(rows["se"].iloc[0])
    if len(values) == 1:
        return {"status": OK, "value": values[0], "unit": unit, "how": f"'{rows['se'].iloc[0]}' 행", "note": note}
    if len(values) > 1:
        return {"status": UNVERIFIED, "value": None, "unit": unit, "how": f"값 여러 개 {values}",
                "note": f"같은 항목 값이 여러 개 {values}: 원문 확인"}
    return {"status": NO_DATA, "value": None, "unit": unit, "how": f"값이 비어 있음 {raw_values}", "note": note}


def find_in_tables(files: dict[str, str], label: str, as_text: bool = False) -> Fetched:
    """원문 표에서 첫 칸(또는 앞 칸)에 label이 들어간 행을 찾아, 그 뒤 첫 숫자(as_text면 첫 글자 칸)를 꺼낸다.

    DART 원문 XML의 TABLE/TR/TD·TE·TU와 HTML의 table/tr/td·th를 모두 읽는다.
    후보가 하나로 정해지면 '원문에서 찾음', 여러 개거나 없으면 '확인 불가'.
    """
    key = strip_brackets(label)
    candidates = []
    for name, text in files.items():
        for cells in table_rows(text):
            for i, cell in enumerate(cells[:-1]):
                if key and key in strip_brackets(cell):
                    rest = [c for c in cells[i + 1:] if c]
                    if as_text:
                        value = next((c for c in rest if c not in ("-",)), None)
                    else:
                        value = next((v for v in (to_number(c) for c in rest) if v is not None), None)
                    if value is not None:
                        candidates.append((value, unit_from_label(cell), name, " | ".join(cells)))
                    break
    distinct = sorted({(c[0], c[1]) for c in candidates}, key=str)
    if len(distinct) == 1:
        value, unit, name, row = candidates[0]
        return Fetched(value, FROM_ORIGINAL, Source("document", note=f"{name}: {row}"),
                       [f"원문 표 '{label}' 행"], unit=unit)
    if distinct:
        rows = "; ".join(c[3] for c in candidates[:5])
        return Fetched(None, UNVERIFIED, Source("document", note=f"후보 여러 개: {rows}"),
                       [f"원문 표 '{label}' 후보 {len(distinct)}개"])
    return Fetched(None, UNVERIFIED, Source("document", note=f"원문 표에 '{label}' 행 없음"),
                   [f"원문 표 '{label}' 행 없음"])


def parse_financial_statements(files: dict[str, str]) -> pd.DataFrame:
    """감사보고서 원문에서 재무제표 표를 읽는다. 금액은 원 단위로 바꾼다(단위를 못 찾으면 unit이 비어 있음).

    - 표 앞 글자(또는 표 첫 행)에 재무제표 제목이 하나만 있으면 그 표부터 그 재무제표로 본다.
      감사의견 문단처럼 제목이 여러 개 나열된 글은 무시한다. '주석'이 나오면 재무제표 구간이 끝난다.
    - '(단위 : 원)' · 천원 · 백만원을 읽어 곱한다.
    - 머리 행의 둘째 칸이 '주석'이면 각 행의 둘째 칸(주석 번호)을 뺀다.
    - 남은 금액 칸을 반으로 나눠 앞쪽을 당기, 뒤쪽을 전기로 보고 각 쪽의 마지막 숫자를 쓴다
      (한 기간에 '내역·합계' 두 칸을 쓰는 표도 읽힘). 칸 수가 홀수면 당기·전기를 정할 수 없어 비워 둔다.
    """
    rows = []
    for name, text in files.items():
        current, unit, prev_end = None, None, 0
        for m in re.finditer(r"<table\b.*?</table>", text, flags=re.S | re.I):
            # 표 앞 글자를 문단 단위로 나눠 표에 가까운 문단부터 본다
            pieces = [squash(html.unescape(x)) for x in re.split(r"<[^>]+>", text[prev_end:m.start()])]
            pieces = [x for x in pieces if x][::-1]
            prev_end = m.end()
            cells_rows = table_rows(m.group(0))
            head = squash(" ".join(cells_rows[0])) if cells_rows else ""
            title = next((t for t in map(_statement_title, pieces) if t), None)
            if not title and len(cells_rows) <= 3:
                title = _statement_title(head)
            if title:
                current, unit = title, None
            elif current and any(x.startswith("주석") and len(x) <= 15 or "재무제표에대한주석" in x for x in pieces):
                current = None
            found_unit = next((u for u in map(_unit_in, pieces) if u), None) or _unit_in(
                " ".join(squash(" ".join(r)) for r in cells_rows[:3]))
            if found_unit:
                unit = found_unit
            if not current:
                continue
            header = next((r for r in cells_rows if any(k in squash(c) for c in r for k in ("당기", "(당)", "당)"))), None)
            drop_note = bool(header and len(header) > 1 and "주석" in squash(header[1]))
            for cells in cells_rows:
                if cells is header or not cells[0] or _statement_title(squash(cells[0])) or _unit_in(squash(cells[0])):
                    continue
                amounts = cells[2:] if drop_note and len(cells) > 2 else cells[1:]
                if not any(to_number(c) is not None for c in amounts):
                    continue
                if len(amounts) == 1:
                    cur, prev = to_number(amounts[0]), None
                elif len(amounts) % 2 == 0:
                    half = len(amounts) // 2
                    cur = next((v for v in reversed([to_number(c) for c in amounts[:half]]) if v is not None), None)
                    prev = next((v for v in reversed([to_number(c) for c in amounts[half:]]) if v is not None), None)
                else:
                    cur = prev = None
                rows.append({
                    "sj_div": current, "account_nm": cells[0],
                    "thstrm_amount": None if cur is None or unit is None else cur * unit,
                    "frmtrm_amount": None if prev is None or unit is None else prev * unit,
                    "unit": unit, "file": name, "row": " | ".join(cells),
                })
    return pd.DataFrame(rows, columns=["sj_div", "account_nm", "thstrm_amount", "frmtrm_amount", "unit", "file", "row"])


_TITLES = (("포괄손익계산서", "CIS"), ("손익계산서", "IS"), ("재무상태표", "BS"), ("대차대조표", "BS"),
           ("현금흐름표", "CF"), ("자본변동표", "SCE"))


def _statement_title(text: str) -> str | None:
    """글 속 재무제표 제목 하나 → sj_div. 제목이 없거나 여러 종류가 나열돼 있으면 None."""
    found = set()
    rest = text
    for title, code in _TITLES:
        if title in rest:
            found.add(code)
            rest = rest.replace(title, "")
    return found.pop() if len(found) == 1 else None


def _unit_in(text: str) -> int | None:
    m = re.search(r"단위[:：]?(백만원|천원|원)", squash(text))
    return {"원": 1, "천원": 1_000, "백만원": 1_000_000}[m.group(1)] if m else None


def _source_row(df: pd.DataFrame, spec: Account, column: str, value) -> tuple[str, int]:
    """찾은 값의 원문 행과 표 단위."""
    names = {strip_brackets(n) for n in spec.names}
    hit = df[df["sj_div"].isin(spec.statements) & df["account_nm"].map(strip_brackets).isin(names)]
    hit = hit[hit[column].map(to_number) == value]
    if hit.empty:
        return "", 1
    return f"{hit['file'].iloc[0]}: {hit['row'].iloc[0]}", int(hit["unit"].iloc[0])


MATCH = "일치"
MISMATCH = "불일치"


def cross_check(audit: Fetched, api: Fetched) -> str:
    """감사보고서 값과 사업보고서 API 값 대조. 감사보고서 표가 천원·백만원 단위면 그 단위 안의 차이는 일치로 본다."""
    if audit.value is None:
        return "감사보고서 값 없음"
    if api.value is None:
        return "API 값 없음"
    tolerance = (audit.source.params.get("표 단위", 1) if audit.source else 1) - 1
    gap = abs(audit.value - api.value)
    return MATCH if gap <= tolerance else f"{MISMATCH}(차이 {gap:,}원)"


def table_rows(markup: str) -> list[list[str]]:
    """표 행마다 칸 글자 목록. 태그는 지우고 공백은 하나로 줄인다."""
    rows = []
    for tr in re.findall(r"<tr\b[^>]*>(.*?)</tr>", markup, flags=re.S | re.I):
        cells = re.findall(r"<t[dheu]\b[^>]*>(.*?)</t[dheu]>", tr, flags=re.S | re.I)
        cleaned = [re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", c))).strip() for c in cells]
        if any(cleaned):
            rows.append(cleaned)
    return rows


def match_stock_codes(corp_df: pd.DataFrame, stock_codes) -> pd.DataFrame:
    """종목코드 → 고유번호. 앞자리 0이 빠진 코드도 6자리로 맞춘다. 없거나 여러 개면 '확인 불가'."""
    listed = corp_df[corp_df["stock_code"].astype(str).str.strip() != ""].copy()
    listed["stock_code"] = listed["stock_code"].astype(str).str.strip().str.zfill(6)
    rows = []
    for code in stock_codes:
        norm = re.sub(r"\D", "", str(code)).zfill(6)
        hit = listed[listed["stock_code"] == norm]
        if len(hit) == 1:
            rows.append({"stock_code": norm, "corp_code": hit["corp_code"].iloc[0], "corp_name": hit["corp_name"].iloc[0],
                         "상태": OK, "메모": ""})
        else:
            memo = "고유번호표에 없음(상장폐지·코드 변경 확인)" if hit.empty else f"후보 {len(hit)}개"
            rows.append({"stock_code": norm, "corp_code": None, "corp_name": None, "상태": UNVERIFIED, "메모": memo})
    return pd.DataFrame(rows, columns=["stock_code", "corp_code", "corp_name", "상태", "메모"])


# --- 내부 도우미 ---


def _number_result(raw, src: Source, attempts: list[str], absolute: bool = False) -> Fetched:
    value = to_number(raw)
    if value is None:
        return Fetched(None, NO_DATA, _with_note(src, f"값이 비어 있음({raw!r})"), attempts)
    if absolute and value < 0:
        return Fetched(-value, OK, _with_note(src, "음수로 와서 절댓값으로 저장"), attempts)
    return Fetched(value, OK, src, attempts)


def _with_note(src: Source | None, note: str) -> Source | None:
    if src is None:
        return Source("", note=note)
    joined = f"{src.note} · {note}" if src.note else note
    return Source(src.api, src.params, src.rcept_no, src.url, joined)


def _json_status(raw: bytes) -> str | None:
    try:
        return str(json.loads(raw).get("status"))
    except (ValueError, AttributeError):
        return None


def _xml_status(raw: bytes) -> str | None:
    m = re.search(rb"<status>\s*(\d+)\s*</status>", raw)
    return m.group(1).decode() if m else None


def _decode(data: bytes) -> str:
    for enc in ("utf-8", "cp949"):
        try:
            return data.decode(enc)
        except UnicodeDecodeError:
            continue
    return data.decode("utf-8", errors="replace")


def _ymd(value) -> str:
    if isinstance(value, (date, datetime)):
        return value.strftime("%Y%m%d")
    return re.sub(r"\D", "", str(value))


def _quarters(start: str, end: str) -> list[tuple[str, str]]:
    """YYYYMMDD 기간을 3개월 이하 조각으로 나눈다."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out = []
    while s <= e:
        stop = min(s + pd.DateOffset(months=3) - pd.Timedelta(days=1), e)
        out.append((s.strftime("%Y%m%d"), stop.strftime("%Y%m%d")))
        s = stop + pd.Timedelta(days=1)
    return out

In [ ]:
%%writefile core/sources/samples.py
"""인터넷·키 없이 결측 보완 흐름을 보여 주는 가짜 응답. 테스트와 팀 공유 노트북이 함께 쓴다.

회사 '00000001'(가상 회사 가나다)은 실제 응답에서 자주 보던 결측 모양을 일부러 담았다.
- 연결재무제표 없음(013) → 별도만 있음, 계정 이름이 '당기순이익(손실)', account_id가 '-표준계정코드 미사용-'
- 2024 사업보고서 배당 표: 현금배당금총액 행이 '-' , 주당 현금배당금은 보통주·우선주 두 행
- 2025 사업보고서 배당 표의 전기(frmtrm) 칸에 2024 현금배당금총액이 있음
- 배당 결정 공시 원문: DART XML 표(TD·TE)로만 숫자가 있음
- 기업가치 제고 계획 공시: 원문 파일 없음(014)
- 감사보고서(별도) 2024·2025: 단위 천원, 둘째 칸이 주석 번호, 한 기간에 금액 칸 두 개(내역·합계),
  감사의견 문단에 재무제표 이름이 나열됨, 주석 표에도 '부채총계'가 있음. 2023 감사보고서는 없음.
  2025 자본총계는 사업보고서 API 값과 일부러 다르게 넣음(대조 불일치 예시)

회사 '00000002'(가상 회사 라마바)는 연결 작성 회사다.
- 2025 연결감사보고서: 단위 백만원, 포괄손익계산서에 '지배기업 소유주지분'이 순이익 귀속·총포괄이익 귀속 두 번 나옴
- 2025 사업보고서 연결 API: account_id가 있음
숫자는 모두 가상이다.
"""
import io
import json
import zipfile

import pandas as pd

CORP = "00000001"
STOCK = "012345"
CORP2 = "00000002"
AUDIT_2024 = "20250310000901"
AUDIT_2025 = "20260309000902"
CONSOL_AUDIT_2025 = "20260311000903"
DECISION_RCEPT = "20260210800123"
VALUEUP_RCEPT = "20250530800456"


def _ok(rows, **extra) -> bytes:
    return json.dumps({"status": "000", "message": "정상", "list": rows, **extra}, ensure_ascii=False).encode()


NO_DATA = json.dumps({"status": "013", "message": "조회된 데이타가 없습니다."}, ensure_ascii=False).encode()
LIMIT = json.dumps({"status": "020", "message": "요청 제한을 초과하였습니다."}, ensure_ascii=False).encode()


def _statement_rows(year: int, net: str, prev: str, liab: str, eq: str, rcept: str) -> list[dict]:
    common = {"rcept_no": rcept, "bsns_year": str(year), "corp_code": CORP, "currency": "KRW"}
    return [
        {**common, "sj_div": "BS", "account_id": "ifrs-full_Liabilities", "account_nm": "부채총계",
         "thstrm_amount": liab, "frmtrm_amount": "", "ord": "1"},
        {**common, "sj_div": "BS", "account_id": "ifrs-full_Equity", "account_nm": "자본총계",
         "thstrm_amount": eq, "frmtrm_amount": "", "ord": "2"},
        {**common, "sj_div": "IS", "account_id": "-표준계정코드 미사용-", "account_nm": "당기순이익(손실)",
         "thstrm_amount": net, "frmtrm_amount": prev, "ord": "3"},
        {**common, "sj_div": "CF", "account_id": "dart_DividendsPaid", "account_nm": "배당금의 지급",
         "thstrm_amount": "(1,500,000,000)", "frmtrm_amount": "", "ord": "4"},
    ]


def _dividend_rows(rcept: str, cash_total: tuple[str, str, str]) -> list[dict]:
    t, f, lw = cash_total
    base = {"rcept_no": rcept, "corp_code": CORP, "corp_name": "가나다"}
    return [
        {**base, "se": "(별도)당기순이익(백만원)", "stock_knd": "", "thstrm": "5,000", "frmtrm": "4,000", "lwfr": "3,000"},
        {**base, "se": "현금배당금총액(백만원)", "stock_knd": "", "thstrm": t, "frmtrm": f, "lwfr": lw},
        {**base, "se": "(별도)현금배당성향(%)", "stock_knd": "", "thstrm": "30.0", "frmtrm": "-", "lwfr": "-"},
        {**base, "se": "주당 현금배당금(원)", "stock_knd": "보통주", "thstrm": "500", "frmtrm": "450", "lwfr": "400"},
        {**base, "se": "주당 현금배당금(원)", "stock_knd": "우선주", "thstrm": "505", "frmtrm": "455", "lwfr": "405"},
    ]


DECISION_XML = """<?xml version="1.0" encoding="utf-8"?>
<DOCUMENT><BODY><TABLE>
<TR><TD>1. 배당구분</TD><TD>결산배당</TD></TR>
<TR><TD>2. 배당종류</TD><TD>현금배당</TD></TR>
<TR><TD ROWSPAN="2">3. 1주당 배당금(원)</TD><TD>보통주식</TD><TE>1,500</TE></TR>
<TR><TD>종류주식</TD><TE>1,505</TE></TR>
<TR><TD ROWSPAN="2">4. 시가배당율(%)</TD><TD>보통주식</TD><TE>3.2</TE></TR>
<TR><TD>종류주식</TD><TE>3.4</TE></TR>
<TR><TD>5. 배당금총액(원)</TD><TE>1,512,345,000</TE></TR>
<TR><TD>6. 배당기준일</TD><TE>2025-12-31</TE></TR>
</TABLE></BODY></DOCUMENT>"""


def _zip(files: dict[str, str]) -> bytes:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as z:
        for name, text in files.items():
            z.writestr(name, text)
    return buf.getvalue()


CORPCODE_XML = f"""<?xml version="1.0" encoding="UTF-8"?>
<result>
<list><corp_code>{CORP}</corp_code><corp_name>가나다</corp_name><stock_code>{STOCK}</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000002</corp_code><corp_name>라마바</corp_name><stock_code>067890</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000003</corp_code><corp_name>비상장사</corp_name><stock_code> </stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000004</corp_code><corp_name>인지디스플레이</corp_name><stock_code>037330</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000005</corp_code><corp_name>현대자동차</corp_name><stock_code>005380</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000006</corp_code><corp_name>GS</corp_name><stock_code>078930</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000007</corp_code><corp_name>GS리테일</corp_name><stock_code>007070</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000008</corp_code><corp_name>신영증권</corp_name><stock_code>001720</stock_code><modify_date>20260101</modify_date></list>
<list><corp_code>00000009</corp_code><corp_name>인지디스플레이테크</corp_name><stock_code>099990</stock_code><modify_date>20260101</modify_date></list>
</result>"""

# 기업개황(company.json): 법인구분 Y 유가·K 코스닥, 결산월, 업종코드(64~66 금융업)
COMPANY = {
    CORP: ("가나다", "가나다", STOCK, "K", "264", "12"),
    "00000002": ("라마바", "라마바", "067890", "Y", "301", "12"),
    "00000004": ("인지디스플레이", "인지디스플레", "037330", "K", "262", "12"),
    "00000005": ("현대자동차", "현대차", "005380", "Y", "301", "12"),
    "00000006": ("GS", "GS", "078930", "Y", "701", "12"),
    "00000007": ("GS리테일", "GS리테일", "007070", "Y", "471", "12"),
    "00000008": ("신영증권", "신영증권", "001720", "Y", "661", "03"),
    "00000009": ("인지디스플레이테크", "인지디스테크", "099990", "Y", "262", "12"),  # 이름은 비슷하지만 유가
}


_HEAD = ("<TR><TD>과 목</TD><TD>주 석</TD><TD>제 {n}(당) 기</TD><TD></TD>"
         "<TD>제 {p}(전) 기</TD><TD></TD></TR>")


def _audit_xml(n: int, debt: str, equity: str, net: str, prev_net: str, paid: str) -> str:
    head = _HEAD.format(n=n, p=n - 1)
    return f"""<?xml version="1.0" encoding="utf-8"?>
<DOCUMENT><BODY>
<P>독립된 감사인의 감사보고서</P>
<P>우리는 재무상태표, 손익계산서, 자본변동표, 현금흐름표 그리고 주석으로 구성된 재무제표를 감사하였습니다.</P>
<P>재 무 상 태 표</P><P>제 {n} 기 {2014 + n}년 12월 31일 현재</P><P>(단위 : 천원)</P>
<TABLE>{head}
<TR><TD>자산총계</TD><TD></TD><TD></TD><TD>30,000,000</TD><TD></TD><TD>28,000,000</TD></TR>
<TR><TD>부채총계</TD><TD>4,5</TD><TD></TD><TD>{debt}</TD><TD></TD><TD>9,000,000</TD></TR>
<TR><TD>자본총계</TD><TD>17</TD><TD></TD><TD>{equity}</TD><TD></TD><TD>19,000,000</TD></TR>
</TABLE>
<P>첨부된 주석은 본 재무제표의 일부입니다.</P>
<P>손 익 계 산 서</P><P>(단위 : 천원)</P>
<TABLE>{head}
<TR><TD>Ⅰ. 매출액</TD><TD>20</TD><TD></TD><TD>50,000,000</TD><TD></TD><TD>45,000,000</TD></TR>
<TR><TD>Ⅹ. 당기순이익(손실)</TD><TD></TD><TD></TD><TD>{net}</TD><TD></TD><TD>{prev_net}</TD></TR>
</TABLE>
<P>현 금 흐 름 표</P><P>(단위 : 천원)</P>
<TABLE>{head}
<TR><TD>배당금의 지급</TD><TD>25</TD><TD>{paid}</TD><TD></TD><TD>(1,200,000)</TD><TD></TD></TR>
</TABLE>
<P>주석</P><P>1. 일반사항</P>
<TABLE><TR><TD>부채총계</TD><TD>999</TD></TR></TABLE>
</BODY></DOCUMENT>"""


CONSOL_AUDIT_XML = """<?xml version="1.0" encoding="utf-8"?>
<DOCUMENT><BODY>
<P>연 결 재 무 상 태 표</P><P>(단위 : 백만원)</P>
<TABLE><TR><TD>과 목</TD><TD>주 석</TD><TD>당기</TD><TD>전기</TD></TR>
<TR><TD>부채총계</TD><TD>10</TD><TD>8,000</TD><TD>7,500</TD></TR>
<TR><TD>지배기업 소유주지분</TD><TD></TD><TD>11,000</TD><TD>10,000</TD></TR>
<TR><TD>비지배지분</TD><TD></TD><TD>1,000</TD><TD>900</TD></TR>
<TR><TD>자본총계</TD><TD></TD><TD>12,000</TD><TD>10,900</TD></TR>
</TABLE>
<P>연 결 포 괄 손 익 계 산 서</P><P>(단위 : 백만원)</P>
<TABLE><TR><TD>과 목</TD><TD>주 석</TD><TD>당기</TD><TD>전기</TD></TR>
<TR><TD>당기순이익</TD><TD></TD><TD>2,000</TD><TD>1,800</TD></TR>
<TR><TD>지배기업의 소유주에게 귀속되는 당기순이익</TD><TD></TD><TD>1,700</TD><TD>1,500</TD></TR>
<TR><TD>비지배지분에 귀속되는 당기순이익</TD><TD></TD><TD>300</TD><TD>300</TD></TR>
<TR><TD>총포괄이익</TD><TD></TD><TD>2,200</TD><TD>1,900</TD></TR>
<TR><TD>지배기업 소유주지분</TD><TD></TD><TD>1,850</TD><TD>1,580</TD></TR>
</TABLE>
</BODY></DOCUMENT>"""


def _consol_rows() -> list[dict]:
    common = {"rcept_no": "20260316000333", "bsns_year": "2025", "corp_code": CORP2, "currency": "KRW"}
    rows = [
        ("BS", "ifrs-full_Liabilities", "부채총계", "8,000,000,000"),
        ("BS", "ifrs-full_EquityAttributableToOwnersOfParent", "지배기업 소유주지분", "11,000,000,000"),
        ("BS", "ifrs-full_Equity", "자본총계", "12,000,000,000"),
        ("CIS", "ifrs-full_ProfitLoss", "당기순이익", "2,000,000,000"),
        ("CIS", "ifrs-full_ProfitLossAttributableToOwnersOfParent", "지배기업 소유주지분", "1,700,000,000"),
        ("CIS", "ifrs-full_ComprehensiveIncomeAttributableToOwnersOfParent", "지배기업 소유주지분", "1,850,000,000"),
    ]
    return [{**common, "sj_div": sj, "account_id": i, "account_nm": n, "thstrm_amount": v, "frmtrm_amount": ""}
            for sj, i, n, v in rows]


def _audit_list(corp: str, detail: str) -> bytes:
    rows = []
    if corp == CORP and detail == "F001":
        rows = [
            {"corp_code": CORP, "report_nm": "감사보고서 (2024.12)", "rcept_no": AUDIT_2024, "rcept_dt": "20250310"},
            {"corp_code": CORP, "report_nm": "감사보고서 (2025.12)", "rcept_no": AUDIT_2025, "rcept_dt": "20260309"},
        ]
    if corp == CORP2 and detail == "F002":
        rows = [{"corp_code": CORP2, "report_nm": "연결감사보고서 (2025.12)", "rcept_no": CONSOL_AUDIT_2025,
                 "rcept_dt": "20260311"}]
    return _ok(rows, page_no=1, total_page=1, total_count=len(rows)) if rows else NO_DATA


def fake_dart_fetch(url: str, params: dict) -> bytes:
    """OpenDART 주소·파라미터를 보고 가짜 응답을 고른다. 모르는 요청은 013."""
    endpoint = url.rsplit("/", 1)[-1]
    year = str(params.get("bsns_year", ""))
    if endpoint == "corpCode.xml":
        return _zip({"CORPCODE.xml": CORPCODE_XML})
    if endpoint == "company.json":
        if params["corp_code"] not in COMPANY:
            return NO_DATA
        name, short, code, cls, induty, month = COMPANY[params["corp_code"]]
        return json.dumps({"status": "000", "message": "정상", "corp_name": name, "stock_name": short, "stock_code": code,
                           "corp_cls": cls, "induty_code": induty, "acc_mt": month}, ensure_ascii=False).encode()
    if endpoint == "list.json" and params.get("pblntf_ty") == "F":
        return _audit_list(params.get("corp_code"), params.get("pblntf_detail_ty"))
    if endpoint == "document.xml":
        audits = {AUDIT_2024: _audit_xml(10, "10,000,000", "20,000,000", "4,800,000", "3,900,000", "(1,500,000)"),
                  AUDIT_2025: _audit_xml(11, "11,000,000", "21,500,000", "5,100,000", "4,750,000", "(1,500,000)"),
                  CONSOL_AUDIT_2025: CONSOL_AUDIT_XML}
        if params["rcept_no"] in audits:
            return _zip({f"{params['rcept_no']}.xml": audits[params["rcept_no"]]})
    if params.get("corp_code") == CORP2:
        if endpoint == "fnlttSinglAcntAll.json" and params["fs_div"] == "CFS" and year == "2025":
            return _ok(_consol_rows())
        return NO_DATA
    if params.get("corp_code") not in (CORP, None):
        return NO_DATA
    if endpoint == "fnlttSinglAcntAll.json":
        if params["fs_div"] == "CFS":
            return NO_DATA  # 자회사가 없어 연결재무제표가 없다
        if year == "2024":
            return _ok(_statement_rows(2024, "4,800,000,000", "3,900,000,000", "10,000,000,000", "20,000,000,000",
                                       "20250315000111"))
        if year == "2025":
            return _ok(_statement_rows(2025, "5,100,000,000", "4,750,000,000", "11,000,000,000", "21,000,000,000",
                                       "20260316000222"))
        return NO_DATA
    if endpoint == "fnlttSinglAcnt.json":
        return NO_DATA
    if endpoint == "alotMatter.json":
        if year == "2024":
            return _ok(_dividend_rows("20250315000111", ("-", "1,200", "1,000")))
        if year == "2025":
            return _ok(_dividend_rows("20260316000222", ("1,512", "1,400", "-")))
        return NO_DATA
    if endpoint == "stockTotqySttus.json":
        rows = [
            {"rcept_no": "20260316000222", "se": "보통주", "istc_totqy": "1,000,000", "tesstk_co": "8,000", "distb_stock_co": "-"},
            {"rcept_no": "20260316000222", "se": "우선주", "istc_totqy": "10,000", "tesstk_co": "0", "distb_stock_co": "10,000"},
            {"rcept_no": "20260316000222", "se": "합계", "istc_totqy": "1,010,000", "tesstk_co": "8,000", "distb_stock_co": "-"},
        ]
        return _ok(rows)
    if endpoint == "hyslrSttus.json":
        rows = [
            {"rcept_no": "20260316000222", "nm": "홍길동", "relate": "본인", "stock_knd": "보통주", "trmend_posesn_stock_qota_rt": "30.10"},
            {"rcept_no": "20260316000222", "nm": "홍길순", "relate": "친인척", "stock_knd": "보통주", "trmend_posesn_stock_qota_rt": "5.25"},
            {"rcept_no": "20260316000222", "nm": "홍길동", "relate": "본인", "stock_knd": "우선주", "trmend_posesn_stock_qota_rt": "1.00"},
        ]
        return _ok(rows)
    if endpoint == "list.json":
        rows = [
            {"corp_code": CORP, "corp_name": "가나다", "report_nm": "현금ㆍ현물배당결정", "rcept_no": DECISION_RCEPT,
             "rcept_dt": "20260210", "rm": "유"},
            {"corp_code": CORP, "corp_name": "가나다", "report_nm": "[기재정정]기업가치 제고 계획(자율공시)",
             "rcept_no": VALUEUP_RCEPT, "rcept_dt": "20250530", "rm": "유"},
            {"corp_code": CORP, "corp_name": "가나다", "report_nm": "사업보고서 (2025.12)", "rcept_no": "20260316000222",
             "rcept_dt": "20260316", "rm": ""},
        ]
        page = int(params.get("page_no", 1))
        return _ok(rows[:2] if page == 1 else rows[2:], page_no=page, total_page=2, total_count=3)
    if endpoint == "document.xml":
        if params["rcept_no"] == DECISION_RCEPT:
            return _zip({f"{DECISION_RCEPT}.xml": DECISION_XML})
        return b'<?xml version="1.0" encoding="UTF-8"?><result><status>014</status><message>file not found</message></result>'
    return NO_DATA


class FakePrices:
    """pykrx 대신 쓰는 가짜 시세. 012345는 1월 6~7일 거래정지, 067890은 1월 8일 상장."""

    name = "가짜 시세"

    def __init__(self):
        self.days = pd.to_datetime(["2026-01-02", "2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08", "2026-01-09"])

    def index(self, code, start, end):
        return pd.DataFrame({"시가": 2500.0, "고가": 2510.0, "저가": 2490.0,
                             "종가": [2500.0, 2510.0, 2490.0, 2520.0, 2530.0, 2540.0], "거래량": 1_000_000}, index=self.days)

    def tickers(self, date, market):
        return {"012345": "가나다"} if market == "KOSDAQ" else {"067890": "라마바"}

    def prices(self, ticker, start, end):
        if ticker == "012345":
            return pd.DataFrame({
                "시가": [10000, 10100, 0, 0, 10300, 10400], "고가": [10100, 10200, 0, 0, 10400, 10500],
                "저가": [9900, 10000, 0, 0, 10200, 10300], "종가": [10000, 10100, 10100, 10100, 10350, 10450],
                "거래량": [500, 600, 0, 0, 700, 800]}, index=self.days)
        if ticker == "067890":
            d = self.days[4:]
            return pd.DataFrame({"시가": [5000, 5100], "고가": [5200, 5200], "저가": [4900, 5000],
                                 "종가": [5100, 5150], "거래량": [9000, 8000]}, index=d)
        return pd.DataFrame()


# 국가법령정보센터 응답: 검색 결과가 하나라 law가 dict, 항이 하나라 dict, 호는 목록
LAW_SEARCH = {"LawSearch": {"law": {"법령일련번호": "999999", "법령명한글": "조세특례제한법", "시행일자": "20260918",
                                    "현행연혁코드": "현행"}}}
LAW_BODY = {"법령": {"기본정보": {"법령명_한글": "조세특례제한법"}, "조문": {"조문단위": [
    {"조문번호": "104", "조문가지번호": "26", "조문여부": "조문", "조문제목": "다른 조문", "조문내용": "제104조의26(다른 조문)"},
    {"조문번호": "104", "조문여부": "전문", "조문내용": "제5절 ..."},
    {"조문번호": "104", "조문가지번호": "27", "조문여부": "조문", "조문제목": "고배당기업 주식 배당소득에 대한 과세특례",
     "조문내용": "제104조의27(고배당기업 주식 배당소득에 대한 과세특례)",
     "항": {"항번호": "①", "항내용": [["① 다음 각 호의 요건을 모두 갖춘 내국법인 ..."]],
            "호": [{"호번호": "1.", "호내용": "1. 가상 요건 하나"}, {"호번호": "2.", "호내용": "2. 가상 요건 둘",
                                                                "목": {"목번호": "가.", "목내용": "가. 가상 세부 요건"}}]}},
]}}}


def fake_law_fetch(url: str, params: dict) -> bytes:
    if params.get("OC") == "wrong":
        return "<html>인증 오류</html>".encode()
    if url.endswith("lawSearch.do"):
        return json.dumps(LAW_SEARCH, ensure_ascii=False).encode()
    return json.dumps(LAW_BODY, ensure_ascii=False).encode()


# KIND '밸류업 고배당기업' 내려받기와 같은 모양(HTML 표, 종목코드 열 없음, 회사명은 KRX 약칭)
KIND_HTML = """<html><body><table>
<tr><th>번호</th><th>시장구분</th><th>회사명</th><th>공시제목</th><th>사업연도</th><th>결산월</th><th>배당성향(%)</th><th>이익배당금증가율(%)</th></tr>
<tr><td>1</td><td>코스닥</td><td>인지디스플레</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>27.76</td><td>52.36</td></tr>
<tr><td>2</td><td>유가</td><td>현대차</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>25.1</td><td>10.2</td></tr>
<tr><td>3</td><td>유가</td><td>GS</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>40.2</td><td>0</td></tr>
<tr><td>4</td><td>유가</td><td>신영증권</td><td>기업가치 제고 계획(자율공시)</td><td>2026</td><td>03</td><td>40.3</td><td>5</td></tr>
<tr><td>5</td><td>코스닥</td><td>가나다</td><td>[정정] 기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>30.0</td><td>12</td></tr>
<tr><td>6</td><td>유가</td><td>라마바</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>45.0</td><td>10</td></tr>
<tr><td>7</td><td>코스닥</td><td>없는회사</td><td>기업가치 제고 계획(자율공시)</td><td>2025</td><td>12</td><td>-</td><td>-</td></tr>
</table></body></html>"""

In [ ]:
# @title 1-2. 불러오기
import os, sys
sys.path.insert(0, os.getcwd())
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from core.sources import samples
from core.sources.base import Client, run_resumable, status_table, secret, forget_secret
print("코드 준비 완료")

## 2. 연결·키 확인 (실패하면 여기서 멈춤)

- 구글 드라이브를 연결하고 내 드라이브 `EXPORT_FOLDER`(없으면 만듦)에 시험 파일을 써 봅니다.
- 진행 상황(이어받기 기록)과 응답 원본은 **내 드라이브** `ONE_PICK_data/{NB_NAME}/`에 둡니다(다음 날 이어서 돌리기 위해).
- 키는 코랩 🔑 **보안 비밀**에 같은 이름으로 넣어 두면 묻지 않습니다. 없으면 화면에 보이지 않는 입력 칸이 뜹니다.
  키는 메모리에만 있고 파일·출력·zip에 남지 않습니다. 잘못 넣었으면 `forget_secret("이름")` 실행 후 이 셀을 다시 실행하세요.

- `DART_API_KEY`(OpenDART 인증키): https://opendart.fss.or.kr → 인증키 신청·관리 (하루 20,000건)

In [ ]:
# @title 2-1. 드라이브 연결 · 저장 폴더 확인
import datetime as _dt
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:  # 코랩이 아닌 곳(시험용)
    IN_COLAB = False

DRIVE = Path("/content/drive/MyDrive") if IN_COLAB else Path("my_drive")
EXPORT = DRIVE / EXPORT_FOLDER
EXPORT.mkdir(parents=True, exist_ok=True)
OUT = DRIVE / "ONE_PICK_data" / NB_NAME

probe = EXPORT / f".쓰기시험_{NB_NAME}"
try:
    probe.write_text("ok"); probe.unlink()
except OSError as error:
    raise RuntimeError(f"저장 폴더에 쓸 수 없습니다(드라이브 용량·연결 확인): {EXPORT} · {error}")
OUT.mkdir(parents=True, exist_ok=True)
RAW = OUT / "raw"
RESULT = OUT / "결과"
RESULT.mkdir(exist_ok=True)
print("결과 zip 폴더:", EXPORT)
print("내 작업 폴더:", OUT)

In [ ]:
# @title 2-2. 키 받기 · 실제로 한 번 불러 확인
secret("DART_API_KEY", "OpenDART 인증키 입력(화면에 보이지 않음): ")
from core.sources.opendart import Dart, cross_check
dart = Dart(client=Client(cache_dir=RAW))
probe = dart.company("00126380")  # 키가 틀리면 여기서 DartKeyError로 멈춤
print("OpenDART 키 확인 완료" if probe.found else f"키는 통과, 시험 조회 상태: {probe.status}")

## 3. 대상 회사 (company_map.csv)

`01_기업_매핑표_만들기` 담당자에게 받은 `company_map.csv`를 씁니다. 다음 순서로 찾고, 없으면 업로드 창을 띄웁니다.
코랩 왼쪽 📁에 직접 올린 파일 → 내 드라이브 `EXPORT_FOLDER` → 내 작업 폴더. `매핑 상태`가 '정상'인 회사만 수집합니다.

In [ ]:
# @title 3-1. 매핑표 읽기
places = [Path(MAP_FILE), Path("/content") / MAP_FILE, EXPORT / MAP_FILE, OUT / MAP_FILE]
map_path = next((p for p in places if p.is_file()), None)
if map_path is None:
    if not IN_COLAB:
        raise FileNotFoundError(f"{MAP_FILE}이 없습니다: {[str(p) for p in places]}")
    from google.colab import files
    print(f"{MAP_FILE}을 찾지 못했습니다. 업로드해 주세요.")
    up = files.upload()
    map_path = OUT / next(iter(up))
    map_path.write_bytes(next(iter(up.values())))
company_map = pd.read_csv(map_path, dtype=str)
targets = company_map[company_map["매핑 상태"] == "정상"].reset_index(drop=True)
skipped = company_map[company_map["매핑 상태"] != "정상"]
print("매핑표:", map_path, "·", len(company_map), "곳 중 대상", len(targets), "곳")
if len(skipped):
    print(f"고유번호가 없어 빠지는 회사 {len(skipped)}곳:", ", ".join(skipped["회사명"].head(20)))

## 4. 수집 (하루 한도에 걸리면 내일 이어서)

- 회사마다 결과를 `결과/회사별/<고유번호>.csv`로 바로 저장하고, 끝난 회사를 진행 기록에 적습니다.
- 하루 한도(020)에 걸리면 멈춥니다. **다음 날 다시 '모두 실행'하면 남은 회사부터** 이어 갑니다(그때도 끝에 zip을 내보냄).
- 한 회사·한 해에 감사보고서 검색·원문 2~3건, 대조용 사업보고서 API 1~3건, 배당 표 1~3건 정도를 요청합니다.

In [ ]:
# @title 4-1. 수집
FS, BASIS = "OFS", "별도"
ACCOUNT_ITEMS = ["당기순이익_전체", "부채총계", "자본총계", "배당금지급_지급기준"]
DIVIDEND_ITEMS = ["현금배당금총액", "주당현금배당금", "현금배당성향", "당기순이익", "주당순이익"]
years = [int(y) for y in YEARS.split(",") if y.strip()]
BY_CORP = RESULT / "회사별"
BY_CORP.mkdir(exist_ok=True)

def collect_one(corp_code):
    items, api_values, checks = {}, [], []
    for y in years:
        for it in ACCOUNT_ITEMS:
            audit = dart.audit_account(corp_code, y, it, FS)
            api = dart.account(corp_code, y, it, fs_divs=(FS,))
            items[f"{y}|감사보고서|{it}"] = audit
            api_values.append(api.won)
            checks.append(cross_check(audit, api))
        for it in DIVIDEND_ITEMS:
            items[f"{y}|배당 표|{it}"] = dart.dividend_item(corp_code, y, it, basis=BASIS)
            api_values.append(None)
            checks.append("")
    t = status_table(items)
    t.insert(2, "원 단위", [f.won for f in items.values()])
    t.insert(3, "사업보고서 API 값", api_values)
    t.insert(4, "대조", checks)
    t[["사업연도", "출처", "항목"]] = t.pop("항목").str.split("|", expand=True)
    t.insert(0, "corp_code", corp_code)
    t.insert(1, "기준", BASIS)
    t["원문"] = [f.source.url if f.source else None for f in items.values()]
    t.to_csv(BY_CORP / f"{corp_code}.csv", index=False, encoding="utf-8-sig")
    return len(t)

run = run_resumable(targets["corp_code"], collect_one, OUT / "진행기록.json")
print("이번에 끝낸 회사:", len(run["results"]), "| 남은 회사:", len(run["remaining"]), "| 요청 수:", dart.client.calls)
if run["stopped_by_limit"]:
    print("⚠️ 하루 한도에 걸려 멈췄습니다. 지금까지 결과는 아래에서 zip으로 내보냅니다. 내일 다시 '모두 실행'하세요.")

In [ ]:
# @title 4-2. 합치기 · 원문 확인 목록
files = sorted(BY_CORP.glob("*.csv"))
result = pd.concat([pd.read_csv(f, dtype=str) for f in files], ignore_index=True) if files else pd.DataFrame()
if result.empty:
    print("아직 수집한 회사가 없습니다.")
else:
    result.insert(1, "회사명", result["corp_code"].map(company_map.set_index("corp_code")["회사명"]))
    result.to_csv(RESULT / "별도_수집결과.csv", index=False, encoding="utf-8-sig")
    memo, check_col = result["메모"].fillna(""), result["대조"].fillna("")
    need = (~result["상태"].isin(["정상", "원문에서 찾음"]) | check_col.str.startswith("불일치")
            | check_col.isin(["API 값 없음", "감사보고서 값 없음"]) | memo.str.contains("전기 값"))
    check = result[need]
    check.to_csv(RESULT / "별도_원문확인목록.csv", index=False, encoding="utf-8-sig")
    print("값", len(result), "개 · 원문 확인", len(check), "개")
    display(result["상태"].value_counts().rename("상태별 개수").to_frame())
    display(check_col[check_col != ""].str.replace(r"\(.*", "", regex=True).value_counts().rename("대조 결과").to_frame())

## 5. 내보내기 (zip → 내 드라이브)

`결과/` 폴더(수집결과·원문확인목록·회사별 CSV)를 zip 하나로 묶어 내 드라이브 `EXPORT_FOLDER`에 `<노트북 이름>_<날짜_시간>.zip`으로 넣습니다. 응답 원본(`raw/`)과 진행 기록은 넣지 않습니다.
팀에 넘길 때는 이 zip을 직접 공유 폴더나 메신저에 올려 주세요.

In [ ]:
# @title 5-1. zip 내보내기
import shutil, tempfile
stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M")
with tempfile.TemporaryDirectory() as tmp:
    made = shutil.make_archive(str(Path(tmp) / f"{NB_NAME}_{stamp}"), "zip", root_dir=RESULT)
    target = EXPORT / Path(made).name
    shutil.copyfile(made, target)
names = sorted(p.relative_to(RESULT).as_posix() for p in RESULT.rglob("*") if p.is_file())
print("내보냄:", target)
print("들어간 파일", len(names), "개:", names[:15], "..." if len(names) > 15 else "")